First look: ดูโครงสร้าง ช่วงเวลา หน่วย และค่าหาย ไม่วิเคราะห์

In [1]:
import requests
import pandas as pd
import truststore; truststore.inject_into_ssl()

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
r = requests.get(URL, timeout=30)
r.status_code, r.headers.get("Content-Type")

(200, 'application/json; charset=UTF-8')

In [2]:
data = r.json()
print(type(data), list(data.keys()) if isinstance(data, dict) else len(data))
stations = data["stations"]   # ถ้า key ไม่ใช่ชื่อนี้ ให้แก้ตามที่เห็น
print(len(stations))
stations[0]

<class 'dict'> ['stations']
178


{'stationID': '119t',
 'nameTH': 'สวนสาธารณะธารา',
 'nameEN': 'Thara Public Park',
 'areaTH': 'ต.ปากน้ำ อ.เมือง, กระบี่',
 'areaEN': 'Pak Nam Subdistrict, Mueang District, Krabi',
 'stationType': 'GROUND',
 'lat': '8.0506237',
 'long': '98.9180489',
 'forecast': [],
 'AQILast': {'date': '2026-11-19',
  'time': '00:00',
  'PM25': {'color_id': '1', 'aqi': '10', 'value': '5.8'},
  'PM10': {'color_id': '0', 'aqi': '-1', 'value': '-1'},
  'O3': {'color_id': '1', 'aqi': '15', 'value': '21'},
  'CO': {'color_id': '1', 'aqi': '2', 'value': '0.30'},
  'NO2': {'color_id': '1', 'aqi': '0', 'value': '1'},
  'SO2': {'color_id': '1', 'aqi': '1', 'value': '2'},
  'AQI': {'color_id': '1', 'aqi': '15', 'param': 'O3'}}}

In [3]:
df_now = pd.json_normalize(stations)
print(df_now.shape)
df_now.columns.tolist()

(178, 32)


['stationID',
 'nameTH',
 'nameEN',
 'areaTH',
 'areaEN',
 'stationType',
 'lat',
 'long',
 'forecast',
 'AQILast.date',
 'AQILast.time',
 'AQILast.PM25.color_id',
 'AQILast.PM25.aqi',
 'AQILast.PM25.value',
 'AQILast.PM10.color_id',
 'AQILast.PM10.aqi',
 'AQILast.PM10.value',
 'AQILast.O3.color_id',
 'AQILast.O3.aqi',
 'AQILast.O3.value',
 'AQILast.CO.color_id',
 'AQILast.CO.aqi',
 'AQILast.CO.value',
 'AQILast.NO2.color_id',
 'AQILast.NO2.aqi',
 'AQILast.NO2.value',
 'AQILast.SO2.color_id',
 'AQILast.SO2.aqi',
 'AQILast.SO2.value',
 'AQILast.AQI.color_id',
 'AQILast.AQI.aqi',
 'AQILast.AQI.param']

1.หนึ่งแถวของ df_now คืออะไร (หนึ่งสถานี ณ เวลาล่าสุด?)
df_now มี 178 แถว 32 คอลัมน์ ตรงกับจำนวนสถานี 178 แห่งใน `data["stations"]`
พบว่าหนึ่งแถวคือหนึ่งสถานี ณ การวัดล่าสุด (ค่าอยู่ใต้ `AQILast`)
ควรตรวจต่อว่า `stationID` ไม่ซ้ำ และสถานีทุกแห่งมีวันที่/เวลาล่าสุดตรงกันหรือไม่

2.คอลัมน์ไหนเก็บรหัสสถานี ชื่อ พิกัด วันที่ และเวลา
- รหัสสถานี: `stationID`
- ชื่อ: `nameTH`, `nameEN` (พื้นที่: `areaTH`, `areaEN`)
- พิกัด: `lat`, `long`
- วันที่: `AQILast.date`
- เวลา: `AQILast.time`

3.ค่า PM2.5 อยู่คอลัมน์ไหน มีทั้งค่าความเข้มข้นและค่า AQI ไหม หน่วยคืออะไร
- ค่าความเข้มข้นอยู่ที่ `AQILast.PM25.value` และค่า AQI อยู่ที่ `AQILast.PM25.aqi` พบว่ามีทั้งสองแบบ (นอกจากนี้มี `AQILast.PM25.color_id`)
- ตัวอย่างสถานีแรก: value = '5.8', aqi = '10'
- หน่วย: ข้อมูลที่ได้มาไม่ได้ระบุหน่วย ควรตรวจกับเอกสารของ Air4Thai ก่อนสรุปว่าเป็น µg/m³

4.ตัวเลขเก็บเป็นข้อความ (object) หรือตัวเลข มีค่าแปลก เช่น ค่าลบ หรือ “-” แทนค่าหายไหม
- พบว่าค่าตัวเลขในข้อมูลดิบเป็นข้อความ (เช่น '5.8', '21', '0.30') เมื่อทำเป็นตารางน่าจะเป็น object ควรยืนยันด้วย `df_now.dtypes`
- พบค่าแปลก: สถานีแรกมี `PM10.value` = '-1' และ `PM10.aqi` = '-1' ซึ่งน่าจะใช้แทนค่าหาย (ไม่ใช่ค่าติดลบจริง) ควรตรวจต่อว่า PM2.5 ของสถานีอื่นมีค่า '-1' หรือ '-' ด้วยหรือไม่ (ดูผล cell 4)
- มีคอลัมน์ `forecast` เป็นลิสต์ว่าง `[]` ในสถานีแรก

In [4]:
pm = pd.to_numeric(df_now["AQILast.PM25.value"], errors="coerce")
print("แปลงเป็นตัวเลขไม่ได้:", pm.isna().sum())
print("ค่าติดลบ:", (pm < 0).sum())
pm.describe()

แปลงเป็นตัวเลขไม่ได้: 0
ค่าติดลบ: 2


count    178.000000
mean      19.134831
std        6.930349
min       -1.000000
25%       16.125000
50%       19.200000
75%       23.175000
max       43.800000
Name: AQILast.PM25.value, dtype: float64